In [3]:
# Import libraries
import cv2 as cv
import numpy as np

# Open the input video file
cap = cv.VideoCapture('videos/result_equalize.mp4')

# Define the cropping region coordinates
y1, y2 = 50, 300   
x1, x2 = 0, 400    

# Calculate the dimensions of the cropped frame
crop_width  = x2 - x1   # 400
crop_height = y2 - y1   # 250

# Get the frame rate (FPS) of the input video
fps = cap.get(cv.CAP_PROP_FPS)

# Define the codec for the output video
fourcc = cv.VideoWriter_fourcc(*'mp4v')

# Create a VideoWriter object to save the cropped video
out = cv.VideoWriter(
    'videos/cropped_video.mp4',   # Output video file path
    fourcc,                       # Video codec
    fps,                          # Frame rate
    (crop_width, crop_height)     # Output frame size
)

# Process video frame by frame
while True:
    # Read the next frame
    ret, frame = cap.read()

    # Exit when no more frames are available
    if not ret:
        print('Exiting...')
        break

    # Crop the selected region from the current frame
    cropped = frame[y1:y2, x1:x2]

    # Write the cropped frame to the output video
    out.write(cropped)

    # Display the cropped frame in a window
    cv.imshow('Cropped Frame', cropped)

    # Exit if the ESC key is pressed
    if cv.waitKey(1) & 0xFF == 27:
        break

# Release video resources
cap.release()
out.release()

# Close all OpenCV windows
cv.destroyAllWindows()

Exiting...


In [5]:
# Import libraries
import cv2 as cv
import numpy as np

# Open the input video file
cap = cv.VideoCapture('videos/cropped_video.mp4')

# Get the frame rate (FPS) of the input video
fps = cap.get(cv.CAP_PROP_FPS)

# Define the codec for the output video
fourcc = cv.VideoWriter_fourcc(*'mp4v')

# Create a VideoWriter object to save the result video
out = cv.VideoWriter(
    'videos/histogram_equalization.mp4',   
    fourcc,                             
    fps,                                 
    (crop_width, crop_height),           
    False                                
)

# Create CLAHE object for adaptive contrast enhancement
clahe = cv.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

# Gamma value for brightness correction
gamma = 0.1

lookUpTable = np.array(
    [np.clip((i/255.0)**gamma * 255.0,0,255) for i in range(256)],  # Build gamma correction lookup table
    dtype=np.uint8
).reshape(1,256)                         # Reshape LUT to OpenCV-compatible format

# Mean intensity threshold for applying gamma correction
threshold = 80

while True:
    # Read the next frame from the video
    ret, frame = cap.read()

    # Stop if no frame is returned (end of video)
    if not ret:
        print('Exiting...')
        break

    # Convert frame to grayscale
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Enhance local contrast using CLAHE
    result = clahe.apply(gray)

    # Check if the frame is too dark
    if np.mean(result) < threshold:
        # Apply gamma correction to brighten the frame
        result = cv.LUT(result, lookUpTable)

    # Write processed frame to the output video
    out.write(result)

    # Display the processed frame in a window
    cv.imshow('Result', result)

    # Exit loop if the ESC key is pressed
    if cv.waitKey(1) & 0xFF == 27:
        break

# Release video resources
cap.release()
out.release()

# Close all OpenCV windows
cv.destroyAllWindows()


Exiting...
